In [1]:
import time

from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait


############################################################
import datetime as dt

import pandas as pd


from src.config.configlog import config, logger


from src.scraper.pages.login import LoginPage
from src.scraper.pages.content import ContentPage
from src.scraper.pages.practice import PracticePage 
from src.scraper.pages.reports import ReportsPage
from src.scraper.pages.visitsReport import VisitsReport
from src.scraper.pages.billing import BillingReport
from src.scraper.pages.cptReport import CPTsReport
from src.scraper.pages.aptReport import AppointmentsReport

from src.scraper.helpers.driver import Driver
from src.scraper.helpers.cleaners import FileCleaner

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
## Instantiate the helper class
helper = FileCleaner()

## Set the directories for downloads and output
downloads = helper.get_dir_path("downloads")
output = helper.get_dir_path("output")

## Set the driver settings
driver_settings = Driver.get_driver(downloads)
driver = driver_settings.driver

## Set the dates for the report
now = dt.datetime.now()
today = now.strftime("%m-%d-%Y")
date_from = config.datefrom

## Instantiate the page objects
login = LoginPage(driver_settings)
content = ContentPage(driver_settings)
reports = ReportsPage(driver_settings)
practice = PracticePage(driver_settings)
# billing = BillingReport(driver_settings)
apts = AppointmentsReport(driver_settings)
visits = VisitsReport(driver_settings)
cptpage = CPTsReport(driver_settings)

## Start the scraping process
driver.get("https://service.emedpractice.com/index.aspx")

## LOGIN
login.enter_username(config.loginid)
login.enter_password(config.loginpassword)
login.click_login()

## GET THE CURRENT CPTS
content.nav_practice()
practice.nav_favorites()
practice.select_crd()
cpts = practice.scrape_cpts()
time.sleep(10)
wd_codes = practice.get_wd_codes(cpts)

## GET CPTs REPORT
content.nav_reports()
reports.load_report("cpt_bills_reportV2")
cptpage.enter_cpt_code(",".join(cpts["code"].unique()))
cptpage.select_search_by("T")
cptpage.click_submit()
cptpage_df = cptpage.scrape_table()
cptpage_df.to_csv("data//downloads//cptreport.csv", index=False)

## GET THE APPOINTMENTS REPORT
content.nav_reports()
reports.load_report("AppointmentReportv1")
apts.select_search_by("T")
apts.click_submit()

apts.download_csv()
time.sleep(5)
helper.rename_csv("AppointmentsReport.csv", "Appointment_Report", downloads)

# ## GET THE BILLING REPORT
# content.nav_reports()
# reports.load_report("BillingServicesReport")
# billing.select_dates(date_from, today)
# billing.run_report()
# time.sleep(5)
# helper.extract_zips(downloads)
# helper.rename_csv("BillingServicesReport.csv", "BillingServicesReport", downloads)

## GET THE VISITS REPORT
reports.load_report("visitreport")
visits.stage_visits(today, today)
visits.click_submit()

visits.download_csv()
time.sleep(5)
helper.rename_csv("VisitReport.csv", "Visit_Report", downloads)
helper.clear_zips(downloads)


# ## GET THE FILES
# billing_csv = helper.get_file("BillingServicesReport.csv", downloads)
# visits_csv = helper.get_file("VisitReport.csv", downloads)


# ## SET THE BILLING DATAFRAME
# billing_cols = ["Chart #", "CPTS IN EMR"]
# billing_group_by_cols = ["Chart #"]
# billing_df = pd.read_csv(billing_csv, on_bad_lines='skip', skiprows=7)
# billing_df = billing_df[billing_cols]

# ## GET THE CPT_COUNTS FOR THE SUMMARY SHEEET
# cpt_counts = billing_df["CPTS IN EMR"].value_counts().reset_index()

# ## GROUP BY CHART NUMBER 
# billing_df = billing_df.groupby(billing_group_by_cols).agg(",".join).reset_index()

# ## GET THE DISTRIBUTION OF CPT COIMBINATIONS
# cpt_combos = billing_df["CPTS IN EMR"].value_counts().reset_index()

# ## SET THE VISITS DATAFRAME
# visits_cols = ["Chart #", "Last Name", "First Name", "Gender", "Last Visit Date", "Visit Count"]
# visits_df = pd.read_csv(visits_csv, on_bad_lines='skip')
# visits_df = visits_df[visits_cols]
# total_visits = visits_df["Visit Count"].sum()

# ## MERGE THE DATAFRAMES FOR FINAL REPORT
# res_df = pd.merge(visits_df, billing_df, on='Chart #', how='left')

# ## Get patient dataframe where CPTs include "PEMHC" and any of the wd_codes
# pe_df = res_df[res_df["CPTS IN EMR"].str.contains("PEMHC", na=False)]
# risky_df = res_df[res_df["CPTS IN EMR"].str.contains("|".join(wd_codes), na=False)]

# ## Get total patients with a "PCAGE" and "PBMHS" CPT
# positives_df = res_df[res_df["CPTS IN EMR"].str.contains("PCAGE") & res_df["CPTS IN EMR"].str.contains("PBMHS")]
# positive_count = len(positives_df)
# summary_df = pd.DataFrame({
#         "Total Patients": [len(res_df)], 
#         "Total Visits": [total_visits],
#         "Total At Risk W/D Patients": [len(risky_df)], 
#         "Total PEMHC Patients": [len(pe_df)],
#         "Total Positive CAGE AIDE & Brief Jail Mental Health Screen": [positive_count]
#         })


# ## SET THE REPORT FILE
# report_file = helper.get_file("report.xlsx", output)


# ## WRITE THE REPORT TO EXCEL
# with pd.ExcelWriter(report_file, engine='xlsxwriter') as writer:
#         res_df.to_excel(writer, sheet_name='Details', index=False)
#         risky_df.to_excel(writer, sheet_name='WD Risk', index=False)
#         pe_df.to_excel(writer, sheet_name='PEMHC', index=False)
#         positives_df.to_excel(writer, sheet_name='Positive CAGE & BMHS', index=False)
#         summary_df.to_excel(writer, sheet_name='Summary', index=False)
#         cpt_combos.to_excel(writer, sheet_name='CPT Distribution', index=False)
#         cpt_counts.to_excel(writer, sheet_name='CPT Counts', index=False)
#         cpts.to_excel(writer, sheet_name='CPT Dictionary', index=False)

# logger.info("Completed Execution")

# driver.close()
# driver.quit()

Driver Set Up


In [5]:
apt_dl = pd.read_csv("data/downloads/AppointmentsReport.csv", skiprows=1, parse_dates=["Appointment Date"])
apt_dl["Appointment Date"] = pd.to_datetime(apt_dl["Appointment Date"], errors='coerce')
apt_dl = apt_dl.loc[apt_dl["Appointment Date"].notnull()]
apt_dl.columns = apt_dl.columns.str.replace('\xa0', '')

apt_dl["Chart#"] = apt_dl["Chart#"].astype(int)

In [6]:
apt_dl

,Appointment Date,Time,CheckInTime,CheckOutTime,AppointmentDuaration,SpecialityName,Facility,Scheduler,Appointment Type,Patient Name,Chart#,Patient DOB,Email,Contact1,Contact2,Appointment Fullfilled,Reason,Appointment Status,Deleted Date,Deleted By,Primary InsuranceName,PolicyNumber,Verified,Verified Status,VerifiedOn,Verified By,Bill#,BillStatus,PlanBegin,PlanEnd,Secondary InsuranceName,Secondary PolicyNumber,Copay,Patient Due,CoIns(%),Deductable,DeductableMet,Created Date,Created By,Unnamed: 39
0,2024-07-12,01:15 AM,12:21:14 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Pililaau-Fisher Oliver,1352,10-26-1986,,(999)-999-9999,,Full filled,C@8----------##**CONTEMPT,,,,,,NaN,,,,2239,Self Pay Bill,,,,,$0.00,$238.00,0%,$0.00,$0.00,07-12-2024 12:19:41,Christie Trang,NaN
1,2024-07-12,01:20 AM,12:24:25 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Woods David,1353,03-27-1962,,(757)-319-9775,,Full filled,##**ABUSE OF FAMILY x3 (FC),,,,,,NaN,,,,2240,Self Pay Bill,,,,,$0.00,$259.64,0%,$0.00,$0.00,07-12-2024 12:22:58,Christie Trang,NaN
2,2024-07-12,01:25 AM,12:29:10 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Kelii Colin,1354,10-25-1963,,(999)-999-9999,,Full filled,C@8------------------##**THEFT 4 HARASSMENT,,,,,,NaN,,,,2241,Self Pay Bill,,,,,$0.00,$238.00,0%,$0.00,$0.00,07-12-2024 12:27:46,Christie Trang,NaN
3,2024-07-12,01:30 AM,02:47:32 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Song Paul,1355,08-10-1984,,(808)-226-2677,,Full filled,##**OVUII DRIVING WHILE REVOKED,,,,,,NaN,,,,2242,Self Pay Bill,,,,,$0.00,$238.00,0%,$0.00,$0.00,07-12-2024 02:45:25,Christie Trang,NaN
4,2024-07-12,01:35 AM,02:53:24 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Wrublewski John,1356,12-27-2002,,(999)-999-9999,,Full filled,##**OVUII,,,,,,NaN,,,,2243,Self Pay Bill,,,,,$0.00,$238.00,0%,$0.00,$0.00,07-12-2024 02:49:35,Christie Trang,NaN
5,2024-07-12,01:40 AM,02:57:12 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,McIntyre Fletcher,1357,05-15-1980,,(999)-999-9999,,Full filled,C@8--------##**URINATION IN PUBLIC,,,,,,NaN,,,,2244,Self Pay Bill,,,,,$0.00,$238.00,0%,$0.00,$0.00,07-12-2024 02:55:34,Christie Trang,NaN
6,2024-07-12,01:45 AM,03:04:53 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Keanini Imai Shea Kylee,350,01-25-1999,,(999)-999-9999,,Full filled,C@8--------##**CONTEMPT,,,,,,NaN,,,,2245,Self Pay Bill,,,,,$0.00,$308.00,0%,$0.00,$0.00,07-12-2024 03:04:42,Christie Trang,NaN
7,2024-07-12,01:50 AM,03:14:56 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD CRD,Arellano Lizbette,1358,11-30-2002,,(999)-999-9999,,Full filled,HARASSMENT LEO,,,,,,NaN,,,,2246,Self Pay Bill,,,,,$0.00,$0.00,0%,$0.00,$0.00,07-12-2024 03:12:33,Christie Trang,NaN
8,2024-07-12,01:55 AM,09:14:47 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD DAY 3,Asiata Philip,1321,12-27-1971,,(999)-999-9999,,Full filled,C@12--------PDD 3 x2,,,,,,NaN,,,,2251,Self Pay Bill,,,,,$0.00,$0.00,0%,$0.00,$0.00,07-12-2024 04:10:28,Christie Trang,NaN
9,2024-07-12,02:00 AM,10:05:54 AM,,,Internal Medicine,Hoala i Ke Ola,Ho ala i Ke Ola,HPD DAY 3,Alsip Jonathan,962,01-03-1988,,(999)-999-9999,,Full filled,ROBBERY 2nd CPO VIO x2,,,,,,NaN,,,,2254,Self Pay Bill,,,,,$0.00,$0.00,0%,$0.00,$0.00,07-12-2024 04:11:20,Christie Trang,NaN


In [ ]:
set(apt_dl["Chart#"]) - set(visit_dl["Chart #"])

{86, 227, 580, 1036, 1361, 1362, 1363}